# LightGBM LambdaRank Baseline

This notebook implements and evaluates a LightGBM ranking model.
LightGBM learns to rank items using user and item features.

In [1]:
import sys
sys.path.append('..')  # Add parent directory to path

import pandas as pd
import numpy as np

from utils.train_test_split import chronological_train_test_split
from models.lgb_ranker import LGBRanker
from utils.evaluation import evaluate_model

## 1. Load Data

In [2]:
# Load the dataset
df = pd.read_csv("../data/data_with_additional_features.csv")

# Ensure timestamp is datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Users: {df['user_id'].nunique():,}")
print(f"Shows: {df['show_id'].nunique():,}")
df.head()

Dataset shape: (12941230, 30)
Date range: 2026-06-01 00:00:00 to 2026-06-30 23:59:00
Users: 100,000
Shows: 13,349


,user_id,playback_session_id,show_id,asset_type,episode_id,day,time,watch_minutes,timestamp,user_total_mins,...,show_median_episode_progression,show_primetime_views,show_global_popularity_log,show_primetime_ratio,show_velocity_delta,seq_decay_affinity_score,seq_immediate_recency_rank,seq_is_favorite_anchor,seq_pop_interaction_leverage,context_primetime_match_score
0,1,1,1,VOD,1.0,3,20:16,54,2026-06-03 20:16:00,9539.0,...,12073.0,2035.0,8.980046,0.256200,0.227082,2.224018,0,0,19.971787,0.050535
1,1,2,1,VOD,2.0,3,11:05,53,2026-06-03 11:05:00,9539.0,...,12073.0,2035.0,8.980046,0.256200,0.227082,2.224018,0,0,19.971787,0.050535
2,1,3,1,VOD,3.0,3,22:46,44,2026-06-03 22:46:00,9539.0,...,12073.0,2035.0,8.980046,0.256200,0.227082,2.224018,0,0,19.971787,0.050535
3,1,4,2,VOD,4.0,1,22:56,1,2026-06-01 22:56:00,9539.0,...,67263.0,71.0,5.673323,0.243986,1.173913,0.036882,0,0,0.209244,0.048126
4,1,5,3,RECORDING,5.0,6,15:26,27,2026-06-06 15:26:00,9539.0,...,23335.0,1058.0,9.083075,0.120159,-0.030859,0.911249,0,0,8.276939,0.023701


## 2. Chronological Train/Validation/Test Split

In [3]:
# Split data chronologically: train / validation / test
train_df, val_df, test_df = chronological_train_test_split(
    df,
    timestamp_col='timestamp',
    test_days=5,
    val_days=5,
    return_splits='train_val_test'
)

Chronological Split Summary:
  Train:      2026-06-01 00:00:00 to 2026-06-20 23:59:00 (8,705,249 rows)
  Validation: 2026-06-20 23:59:00 to 2026-06-25 23:59:00 (2,086,740 rows)
  Test:       2026-06-25 23:59:00 to 2026-06-30 23:59:00 (2,149,241 rows)


## 3. Prepare Train Seen Shows Dictionary

In [4]:
# Create a dictionary of shows each user has seen in training data
train_seen_dict = train_df.groupby('user_id')['show_id'].apply(set).to_dict()

print(f"Users in training with history: {len(train_seen_dict):,}")

Users in training with history: 95,415


## 4. Train LightGBM Ranker

In [5]:
# Initialize and train LightGBM ranker
# Use validation set as the ranking training window
score_col = 'implicit_interaction_score' if 'implicit_interaction_score' in train_df.columns else 'watch_minutes'

lgb_model = LGBRanker(
    candidate_pool_size=500,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=63,
    num_iterations=150
)

lgb_model.fit(
    train_df,        # Full training data for features
    val_df,          # Validation window for ranking training
    user_col='user_id',
    item_col='show_id',
    score_col=score_col
)

 TRAINING LIGHTGBM LAMBDARANK RECOMMENDER

[1/5] Building candidate pool from popular items...
  Candidate pool size: 500

[2/5] Engineering show features...
  Created 11954 show feature profiles

[3/5] Engineering user features...
  Created 95415 user feature profiles

[4/5] Preparing ranking training data...
  Created 1,139,721 training examples
  Positive examples: 381,131
  Negative examples: 758,590

[5/5] Training LightGBM LambdaRank model...
  Model trained with 150 iterations

  Top 5 Most Important Features:
    show_unique_users_log           1091266.2
    show_total_watch_time_log        127634.2
    show_global_popularity_log        89394.7
    show_avg_engagement               73875.3
    user_unique_shows                 37905.9

✓ Training complete!


## 5. View Feature Importance

In [6]:
# Get feature importance
feature_importance = lgb_model.get_feature_importance()
print("\nFeature Importance (Top 10):")
print(feature_importance.head(10).to_string(index=False))


Feature Importance (Top 10):
                    feature   importance
      show_unique_users_log 1.091266e+06
  show_total_watch_time_log 1.276342e+05
 show_global_popularity_log 8.939475e+04
        show_avg_engagement 7.387533e+04
          user_unique_shows 3.790586e+04
  user_show_diversity_match 2.159801e+04
user_show_interaction_ratio 1.651939e+04
    user_total_interactions 1.616844e+04
 popularity_bias_resistance 1.609581e+04
       user_diversity_score 1.397859e+04


## 6. Test Predictions for a Sample User

In [7]:
# Test on a sample user
sample_user = train_df['user_id'].iloc[100]
print(f"Sample user: {sample_user}")

# Get recommendations
recommendations = lgb_model.predict(sample_user, top_n=10)
print(f"\nTop 10 recommendations: {recommendations}")

# Get what they watched in training
user_history = train_df[train_df['user_id'] == sample_user]['show_id'].unique()[:5]
print(f"\nUser's watch history (first 5): {list(user_history)}")

Sample user: 1

Top 10 recommendations: [np.int64(172), np.int64(10), np.int64(25), np.int64(72), np.int64(146), np.int64(222), np.int64(353), np.int64(48), np.int64(252), np.int64(182)]

User's watch history (first 5): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(6)]


## 7. Evaluate on Test Set

In [8]:
# Evaluate on test set
test_metrics = evaluate_model(
    model=lgb_model,
    test_ground_truth=test_df,
    train_seen_dict=train_seen_dict,
    user_col='user_id',
    item_col='show_id',
    top_n=10,
    verbose=True
)

Evaluating model on 77,272 users...
  Processing batch 1/78 (0/77,272 users)...
  Processing batch 11/78 (10,000/77,272 users)...
  Processing batch 21/78 (20,000/77,272 users)...
  Processing batch 31/78 (30,000/77,272 users)...
  Processing batch 41/78 (40,000/77,272 users)...
  Processing batch 51/78 (50,000/77,272 users)...
  Processing batch 61/78 (60,000/77,272 users)...
  Processing batch 71/78 (70,000/77,272 users)...

  Evaluation Results (Top-10)
  Users Evaluated: 77,272
  Recall@10:    20.86%
  MRR@10:       0.2192
  Precision@10: 7.91%



## 8. Final Summary

In [9]:
print("\n" + "="*70)
print(" LIGHTGBM RANKER - FINAL RESULTS")
print("="*70)
print(f"\n{'Metric':<20} {'Test Set':<15}")
print("-"*35)
print(f"{'Recall@10':<20} {test_metrics['recall@10']*100:>14.2f}%")
print(f"{'MRR@10':<20} {test_metrics['mrr@10']:>14.4f}")
print(f"{'Precision@10':<20} {test_metrics['precision@10']*100:>14.2f}%")
print(f"{'Users Evaluated':<20} {test_metrics['num_users_evaluated']:>14,}")
print("="*70)


 LIGHTGBM RANKER - FINAL RESULTS

Metric               Test Set       
-----------------------------------
Recall@10                     20.86%
MRR@10                       0.2192
Precision@10                   7.91%
Users Evaluated              77,272


## 8. Save Model

Save the trained model for later use in the inference pipeline

In [10]:
import pickle
import os

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save the model
model_path = '../models/philo_lgb_recommender.pkl'

with open(model_path, 'wb') as f:
    pickle.dump(lgb_model, f)

print(f'✓ Model saved to: {model_path}')
print(f'  Model type: {type(lgb_model).__name__}')
print(f'  Candidate pool size: {len(lgb_model.candidate_pool):,} shows')
print(f'  Features: {len(lgb_model.feature_cols)}')
print(f'  Performance: Recall@10 = {test_metrics["recall@10"]*100:.2f}%')
print(f'\nYou can now use this model in the inference pipeline!')
print(f'Run: jupyter notebook notebooks/09_inference_pipeline.ipynb')

✓ Model saved to: ../models/philo_lgb_recommender.pkl
  Model type: LGBRanker
  Candidate pool size: 500 shows
  Features: 11
  Performance: Recall@10 = 20.86%

You can now use this model in the inference pipeline!
Run: jupyter notebook notebooks/09_inference_pipeline.ipynb


## 9. Verify Model Can Be Loaded

Quick test to ensure the saved model works

In [11]:
# Test loading the model
print('Testing model load...')

with open(model_path, 'rb') as f:
    loaded_model = pickle.load(f)

print(f'✓ Model loaded successfully!')
print(f'  Model type: {type(loaded_model).__name__}')
print(f'  Candidate pool: {len(loaded_model.candidate_pool):,} shows')

# Test prediction on a sample user
sample_user = test_ground_truth['user_id'].iloc[0]
sample_recs = loaded_model.predict(sample_user, top_n=10)

print(f'\n✓ Model works! Sample recommendations for user {sample_user}:')
print(f'  {sample_recs}')

Testing model load...
✓ Model loaded successfully!
  Model type: LGBRanker
  Candidate pool: 500 shows


NameError: name 'test_ground_truth' is not defined